In [ ]:
import os

import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
import zarr
from plotly.offline import init_notebook_mode
from plotly.subplots import make_subplots
from PyriteUtility.spatial_math import spatial_utilities as su

pio.templates.default = "plotly_dark"
pio.renderers.default = "vscode"
init_notebook_mode(connected=True)

# Read log

control_log_folder_path = os.environ.get("PYRITE_CONTROL_LOG_FOLDERS")
control_log_path = control_log_folder_path + "/drag_dense/"

log_store = zarr.DirectoryStore(path=control_log_path)
log = zarr.open(store=log_store, mode="r")  # r: read only, must exist

horizon_count = 0
while True:
    # read the log
    horizon_log = log[f"horizon_{horizon_count}"]
    horizon_count += 1

    ts_pose_fb_0 = horizon_log["ts_pose_fb_0"][:]
    wrench_0 = horizon_log["wrench_0"][:]
    wrench_time_stamps_0 = horizon_log["wrench_time_stamps_0"][:]
    robot_time_stamps_0 = horizon_log["robot_time_stamps_0"][:]
    action = horizon_log["action"][:]
    action_time_stamps = horizon_log["action_time_stamps"][:]

    # convert wrench from tool frame to base frame
    SE3_WT = su.pose7_to_SE3(ts_pose_fb_0)
    adj_TW_fb_0 = su.SE3_to_adj(su.SE3_inv(SE3_WT))

    wrench_0_W = adj_TW_fb_0.T @ wrench_0[..., np.newaxis]

    # plot
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, subplot_titles=("Px", "Fx", "ControlX")
    )

    fig.add_trace(
        go.Scatter(x=robot_time_stamps_0, y=ts_pose_fb_0[:, 0], name="Px"), row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=wrench_time_stamps_0, y=wrench_0_W[:, 0], name="Fx"), row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=action_time_stamps, y=action[:, 0], name="ControlX"), row=3, col=1
    )

    fig.update_layout(height=900, width=900, title_text=horizon_count)
    fig.show()